# Host the app on Google Colab (free T4) with a public testing link

Runs the actual app -- FastAPI + the Gradio UI (chat, tone rewrite, slides, and the Legal tab), not
an eval script -- on a Colab GPU runtime, and opens an `ngrok` tunnel so you can reach it from your
own browser while the session is alive. Same idea as `colab_legal_eval_qwen_gemma.ipynb`, but that
notebook only answers a fixed question set for scoring; this one gives you the live UI to click
around in.

**What you get**: a public HTTPS URL (e.g. `https://xxxx.ngrok-free.app/ui`) that opens the same
Gradio UI you'd see running the app locally -- General GPT chat, tone rewrite, slide generation,
and (if you attach the corpus) the Legal tab.

## Before running
1. *Runtime -> Change runtime type*: **T4 GPU**.
2. *Secrets* (key icon, left): add **GITHUB_TOKEN** (a token that can read `zananiri/AI-IZ`) and
   **NGROK_AUTHTOKEN** (free at https://dashboard.ngrok.com/get-started/your-authtoken -- sign up,
   copy the token shown there). Turn on *Notebook access* for both.
3. Optional: put **`legal_corpus_vectordb.zip`** (from `notebooks/kaggle_legal_corpus_build.ipynb`)
   on your Google Drive and set `CORPUS_ZIP` below to its path, if you want to test the Legal tab.
   Without it, General GPT / tone rewrite / slides still work fine -- the Legal tab will just show
   "Corpus: not installed" and error on a question.
4. *Runtime -> Run all*. Allow Drive access when asked.

## Session limits, and stopping cleanly
Free Colab sessions disconnect on idle and can be reclaimed at any time (as you've already hit with
GPU quota) -- there is no guaranteed uptime. The tunnel URL is public to anyone who has it (no
login), so don't leave it running unattended or share the link beyond your own testing. Run the
**last cell** ("Stop") when you're done, or just let the session end -- either way the tunnel and
model are gone with it, nothing persists beyond what you saved to Drive.


In [ ]:
BRANCH = "main"                              # the branch to run
OLLAMA_MODEL = "qwen3:14b"                   # general chat + Legal orchestrator (both share it, like setup.sh's default)
CONTEXT_LENGTH = 16384                       # tokens per call (Ollama num_ctx) -- config.yaml's legal default
INSTALL_MINERU = True                        # DOCX/PPTX/XLSX/image ingestion (needs LibreOffice, installed below)
INSTALL_HEAVY_OCR = True                     # paddleocr/paddlepaddle/surya-ocr -- large, slower install, needed
                                              # for scanned-page OCR (both tabs) and any non-PDF attachment
CORPUS_ZIP = None                            # e.g. "/content/drive/MyDrive/AI-IZ/legal_corpus_vectordb.zip";
                                              # None = skip the Legal tab's bulk corpus (other tabs still work)
APP_PORT = 8456                              # docslides.api.main's default (DOCSLIDES_APP_PORT)

DRIVE_DIR = "/content/drive/MyDrive/AI-IZ"


## 1. Drive, GPU, code

In [ ]:
import os, pathlib, subprocess, shutil, json, sys, time, urllib.request
from google.colab import drive, userdata

drive.mount("/content/drive")

def sh(command, check=True, env=None):
    """Runs a shell command with its output streamed live; raises on failure when check=True --
    unlike `!command`, which carries on silently."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

REPO = "/content/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("add the GITHUB_TOKEN secret (key icon, left) with notebook access") from exc
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
sh("git log --oneline -1")


## 2. Python environment

A separate `uv` environment, same reason as the eval notebooks: Colab's own preinstalled numpy/
torch/etc. would otherwise fight the exact versions this project pins (see `pyproject.toml` and
`scripts/patch_mineru.py`'s comments on the magic-pdf/transformers version hazards).

In [ ]:
PY = "/content/venv/bin/python"
if not pathlib.Path(PY).exists():
    sh("pip install -q uv")
    sh("uv venv -q --python 3.11 /content/venv")

extras = "ocr,lang,legal,legal-data" if INSTALL_HEAVY_OCR else "lang,legal,legal-data"
sh(f'uv pip install -q --python {PY} -e ".[{extras}]"')
if INSTALL_MINERU:
    sh(f'uv pip install -q --python {PY} -e ".[ingestion-mineru]"')
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y libreoffice > /dev/null")  # DOCX/PPTX/XLSX -> PDF
sh(f"{PY} scripts/patch_mineru.py")  # required after every fresh ingestion-mineru install -- see its own docstring
sh(f"{PY} --version")


## 3. Ollama, the model, and language-ID data

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")

if subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT,
                              env={**os.environ, "OLLAMA_FLASH_ATTENTION": "1", "OLLAMA_KV_CACHE_TYPE": "q8_0"})
    for _ in range(60):
        if not subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
            break
        time.sleep(2)
sh(f"ollama pull {OLLAMA_MODEL}")
sh("ollama list")

# fastText's language-ID model (models/lid.176.bin) -- scripts/download_models.sh's step for it,
# skipping the OCR/spaCy/Stanza weights that script also fetches (INSTALL_HEAVY_OCR's extras pull
# paddleocr/surya's own weights lazily, on first use, same as a normal local install).
pathlib.Path("models").mkdir(exist_ok=True)
if not pathlib.Path("models/lid.176.bin").exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", "models/lid.176.bin"
    )


## 4. The legal corpus (optional)

Skips cleanly if `CORPUS_ZIP` is `None` -- the Legal tab will show "Corpus: not installed" (the
info line above its chat box) instead of erroring on load; a question there will still fail until a
corpus is installed. Every other tab works regardless.

In [ ]:
if CORPUS_ZIP:
    sh(f'{PY} scripts/legal_data/install_corpus.py "{CORPUS_ZIP}" --replace')
else:
    print("CORPUS_ZIP not set -- skipping. The Legal tab will report its corpus as not installed.")


## 5. Start the app

Both the general chat (`llm:`) and the Legal tab's orchestrator (`legal.orchestrator:`) are pointed
at the same local Ollama instance -- `config/config.yaml`'s own defaults target a vLLM Docker
service that doesn't exist here, so both need the env-var overrides `config.py` reads
(`DOCSLIDES_LLM_*` / `DOCSLIDES_LEGAL_ORCHESTRATOR_*`), the same mechanism `docker-compose.portable.yml`
and `scripts/setup.*` use for a non-NVIDIA-Docker host.

In [ ]:
env = {
    **os.environ,
    "DOCSLIDES_APP_PORT": str(APP_PORT),
    "DOCSLIDES_LLM_BACKEND": "ollama", "DOCSLIDES_LLM_BASE_URL": "http://localhost:11434",
    "DOCSLIDES_LLM_MODEL": OLLAMA_MODEL, "DOCSLIDES_LLM_MAX_MODEL_LEN": str(CONTEXT_LENGTH),
    "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama", "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
    "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": OLLAMA_MODEL, "DOCSLIDES_LEGAL_ORCHESTRATOR_MAX_MODEL_LEN": str(CONTEXT_LENGTH),
}

app_log = open("/content/app.log", "a")
app_proc = subprocess.Popen(
    [PY, "-m", "uvicorn", "docslides.api.main:app", "--host", "0.0.0.0", "--port", str(APP_PORT)],
    cwd=REPO, env=env, stdout=app_log, stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        if urllib.request.urlopen(f"http://localhost:{APP_PORT}/health", timeout=2).status == 200:
            print("app is up")
            break
    except Exception:
        time.sleep(2)
else:
    sh(f"tail -50 /content/app.log")
    raise RuntimeError(f"app did not become healthy -- see /content/app.log above (pid {app_proc.pid})")


## 6. Open the public tunnel

Prints the link to open -- it's `/ui` (the Gradio app is mounted there, per `api/main.py`), not the
bare tunnel URL, which only serves the FastAPI JSON routes.

In [ ]:
sh("pip install -q pyngrok")
from pyngrok import ngrok, conf

try:
    conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")
except Exception as exc:
    raise RuntimeError(
        "add the NGROK_AUTHTOKEN secret (key icon, left): sign up free at "
        "https://dashboard.ngrok.com/get-started/your-authtoken and copy the token shown there"
    ) from exc

ngrok.kill()  # any tunnel left over from a previous run of this cell
# 127.0.0.1, not the bare port: ngrok would resolve a bare port via "localhost", which prefers
# IPv6 (::1) on Colab -- refused, since uvicorn above binds --host 0.0.0.0 (IPv4-only wildcard).
tunnel = ngrok.connect(f"127.0.0.1:{APP_PORT}", "http")
print(f"\nOpen this in your browser:\n\n  {tunnel.public_url}/ui\n")
print("Public to anyone with the link, and only alive for the rest of this Colab session.")


## 7. Stop

Run this when you're done testing -- it's also what a session ending or being reclaimed does for
you automatically, just without the tidy log lines.

In [ ]:
ngrok.kill()
app_proc.terminate()
subprocess.run(["ollama", "stop", OLLAMA_MODEL], capture_output=True)
print("stopped: tunnel closed, app process terminated, model unloaded from Ollama.")
